In [ ]:
import os
import torch
from decord import VideoReader, cpu
import matplotlib.pyplot as plt
from holmesvau.holmesvau_utils import load_model, generate, caption_clip, show_smapled_video
from holmesvau.clip_selection import select_clips, snippets_to_frames, load_gt_segments

mllm_path = 'ppxin321/HolmesVAU-2B'
sampler_path = '/home/emogenai4e/emo/VidAnomalyRetrieval/DescriptionModule/Holmes-VAU-ATS/anomaly_scorer.pth'
device = torch.device('cuda')
model, tokenizer, generation_config, sampler = load_model(mllm_path, sampler_path, device)

In [ ]:
import random

video_path = "/home/emogenai4e/emo/VidAnomalyRetrieval/UCF_Video/Abuse/Abuse028_x264.mp4"
gt_annotation_file = "/home/emogenai4e/emo/VidAnomalyRetrieval/DescriptionModule/VadCLIP/list/Temporal_Anomaly_Annotation.txt"

K = 3
clip_sec = 16.0
snippet_size = 16  # dense_sample_freq inside generate()
select_frames = 12

# Description-pool: all 5 prompts are paraphrases of the same training instruction,
# so any of them is in-distribution. We sample one per clip for paraphrase robustness
# (mirrors how the instruction tuning data was sampled per example).
DESCRIPTION_PROMPTS = [
    "Describe the anomaly events observed in the video.",
    "Could you describe the anomaly events observed in the video?",
    "Could you specify the anomaly events present in the video?",
    "Give a description of the detected anomaly events in this video.",
    "How would you describe the particular anomaly events in the video?",
]

# Video-level: the Description prompt was trained on full videos (in-distribution),
# so we use the canonical first prompt deterministically rather than rotating.
VIDEO_PROMPT = DESCRIPTION_PROMPTS[0]

PROMPT_SEED = 0  # set to None for non-deterministic rotation
_prompt_rng = random.Random(PROMPT_SEED)

def pick_description_prompt():
    """Sample one prompt from the description pool. Reproducible via PROMPT_SEED."""
    return _prompt_rng.choice(DESCRIPTION_PROMPTS)

In [ ]:
# 1) Whole-video description + dense anomaly score (via ATS dense pre-pass)
video_pred, _, video_frame_indices, anomaly_score = generate(
    video_path, VIDEO_PROMPT, model, tokenizer, generation_config, sampler,
    dense_sample_freq=snippet_size, select_frames=select_frames, use_ATS=True,
)
print(f'\n[VIDEO] prompt: {VIDEO_PROMPT!r}')
print('       ', video_pred)

In [ ]:
# 2) Cut clips from the anomaly score and describe each one (random prompt per clip)
vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
fps = float(vr.get_avg_fps())
max_frame = len(vr)

clips_snip = select_clips(anomaly_score, K=K, clip_sec=clip_sec, fps=fps, snippet_size=snippet_size)
clips_frame = snippets_to_frames(clips_snip, snippet_size=snippet_size, max_frame=max_frame)
print('Clips (snippet):', clips_snip)
print('Clips (sec):    ', [(round(s * snippet_size / fps, 2), round(e * snippet_size / fps, 2)) for s, e in clips_snip])

clip_results = []
for i, frame_range in enumerate(clips_frame):
    prompt = pick_description_prompt()
    pred, frame_idx = caption_clip(vr, frame_range, prompt, model, tokenizer, generation_config,
                                   select_frames=select_frames)
    clip_results.append({'snippet_range': clips_snip[i], 'frame_range': frame_range,
                         'frame_indices': frame_idx, 'prompt': prompt, 'caption': pred})
    print(f'\n[CLIP {i}] frames={frame_range}  ({frame_range[0]/fps:.2f}s - {frame_range[1]/fps:.2f}s)')
    print(f'  prompt: {prompt!r}')
    print(f'  pred:   {pred}')

In [ ]:
# 3) Visualize: anomaly score, GT (red), selected clips (orange), sampled frames (green)
gt_segments = load_gt_segments(video_path, gt_annotation_file)
print('GT segments (frame):', gt_segments)

show_smapled_video(vr, video_frame_indices)
for i, r in enumerate(clip_results):
    print(f'\n[CLIP {i}] sampled frames:', r['frame_indices'])
    show_smapled_video(vr, r['frame_indices'])

if anomaly_score is not None:
    plt.figure(figsize=(8, 2))
    plt.plot(anomaly_score)
    for j, (gs, ge) in enumerate(gt_segments):
        plt.axvspan(gs / snippet_size, ge / snippet_size, color='red', alpha=0.25,
                    label='ground truth' if j == 0 else None)
    for j, (cs, ce) in enumerate(clips_snip):
        plt.axvspan(cs, ce, color='orange', alpha=0.2,
                    label='selected clip' if j == 0 else None)
    for idx in video_frame_indices:
        plt.vlines(idx / snippet_size, 0, 1, colors='green', linewidth=0.8)
    plt.ylim(0, 1)
    plt.xlabel('snippet')
    plt.ylabel('anomaly score')
    plt.legend(loc='upper right', fontsize=8)
    plt.show()